In [ ]:
import numpy as np
import os
import pandas as pd
import pandas_gbq
import matplotlib.pyplot as plt

In [ ]:
def filter_cases_controls(df, case_filter, control_filter):
    cases = df.query(case_filter).copy()
    controls = df.query(control_filter).copy()
    return cases, controls

def match_cases_to_controls(df: pd.DataFrame, case_filter: str, control_filter:str , n_matched_controls=20, exclude_files=None, seed=42):

    # Exclude samples included in a file designated for removal (e.g., indiviuduals with a PGV, or in CDRv7)
    if exclude_files:
        excluded_ids = {
            line.strip()
            for filepath in exclude_files
            for line in open(filepath)
            if line.strip()
        }
        df = df[~df["person_id"].astype(str).isin(excluded_ids)].copy()

    # Use the filter_cases_controls() function to obtain a suitable set of cases and controls
    cases, controls = filter_cases_controls(df, case_filter, control_filter)

    # Random number will be a tiebreaker for matching controls
    controls["random_number"] = np.random.default_rng(seed).random(len(controls))

    selected_control_ids = []
    available_controls = controls.copy()

    for _, case in cases.iterrows():

        # The only strict requirement for control matching is sex_at_birth and ancestry_pred
        candidate_controls = available_controls[
            (available_controls["sex_at_birth"] == case["sex_at_birth"]) &
            (available_controls["ancestry_pred"] == case["ancestry_pred"])
        ].copy()

        # If there are no suitable controls matching on sex and ancestry, just do sex
        if len(candidate_controls) == 0:
            candidate_controls = available_controls[(available_controls["sex_at_birth"] == case["sex_at_birth"])].copy()

        # If there are no suitable controls matching on sex and ancestry, just do sex
        if len(candidate_controls) == 0:
            candidate_controls = available_controls[(available_controls["ancestry_pred"] == case["ancestry_pred"])].copy()

        # Calculate Age difference
        candidate_controls["age_diff"] = (candidate_controls["age"] - case["age"]).abs()

        # Do controls share similar smoking habits
        candidate_controls["smoke_exact"] = (candidate_controls["smoking_frequency_scaled"] == case["smoking_frequency_scaled"]).astype(int) # Binary for matching
        candidate_controls["smoke_status"] = ((candidate_controls["smoking_frequency_scaled"] == 0) == (case["smoking_frequency_scaled"] == 0)).astype(int) # Binary for both smoking OR both non-smoking

        # Do controls share similar alcohol habits
        candidate_controls["alcohol_exact"] = (candidate_controls["alcohol_frequency_scaled"] == case["alcohol_frequency_scaled"]).astype(int) # Binary for matching
        candidate_controls["alcohol_status"] = ((candidate_controls["alcohol_frequency_scaled"] == 0) == (case["alcohol_frequency_scaled"] == 0)).astype(int) # Binary for both alcohol OR both no-alcohol

        candidate_controls = candidate_controls.sort_values(
            ["age_diff", "smoke_exact", "smoke_status",
             "alcohol_exact", "alcohol_status", "random_number"],
            ascending=[True, False, False, False, False, True],
            na_position="last"
        )

        selected = candidate_controls.head(n_matched_controls).copy()
        
        selected_control_ids.extend(selected["person_id"])
        available_controls = available_controls.drop(selected.index)

    selected_ids = set(cases["person_id"]) | set(selected_control_ids)

    return df[df["person_id"].isin(selected_ids)].copy()

In [ ]:
replication_breast = match_cases_to_controls(replication_df, "breast_cancer == 1", "pancancer_cancer != 1",n_matched_controls=30)

In [ ]:
replication_familial_breast = match_cases_to_controls(replication_df, "breast_cancer == 1 & fdrs_w_breast_cancer >= 1", "pancancer_cancer != 1",n_matched_controls=150)

In [ ]:
replication_kidney = match_cases_to_controls(replication_df, "kidney_cancer == 1", "pancancer_cancer != 1",n_matched_controls=50)

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

def run_cancer_carrier_fisher(carrier_ids_file: str,cancer_type: str,df: pd.DataFrame):
    
    cancer_col = f"{cancer_type}_cancer"

    # Identify cases and controls directly from the provided dataframe
    analysis_df = df.copy()
    analysis_df["person_id"] = analysis_df["person_id"].astype(str)

    analysis_df["case"] = analysis_df[cancer_col] == 1

    # Load carrier IDs
    carrier_ids = set(pd.read_csv(carrier_ids_file, header=None, dtype=str)[0].str.strip())

    # Define carrier status
    analysis_df["carrier"] = analysis_df["person_id"].isin(carrier_ids)

    # Count carriers and noncarriers among cases and controls
    case_carriers = analysis_df.loc[analysis_df["case"], "carrier"].sum()
    case_noncarriers = analysis_df.loc[analysis_df["case"], "carrier"].eq(False).sum()

    control_carriers = analysis_df.loc[~analysis_df["case"], "carrier"].sum()
    control_noncarriers = analysis_df.loc[~analysis_df["case"], "carrier"].eq(False).sum()

    # Construct contingency table
    table = [
        [case_carriers, case_noncarriers],
        [control_carriers, control_noncarriers]
    ]

    # One-sided Fisher's exact test: enrichment among cases
    odds_ratio, p_value = fisher_exact(table, alternative="greater")

    # Print results
    print(f"Cancer type: {cancer_type}")
    print(f"Cases: {analysis_df['case'].sum():,} ({case_carriers:,} carriers)")
    print(f"Controls: {(~analysis_df['case']).sum():,} ({control_carriers:,} carriers)")
    print(f"Odds ratio: {odds_ratio:.4f}")
    print(f"Fisher's exact p-value: {p_value:.4e}")

    return {
        "cancer_type": cancer_type,
        "n_cases": int(analysis_df["case"].sum()),
        "n_controls": int((~analysis_df["case"]).sum()),
        "case_carriers": int(case_carriers),
        "case_noncarriers": int(case_noncarriers),
        "control_carriers": int(control_carriers),
        "control_noncarriers": int(control_noncarriers),
    }

In [ ]:
run_cancer_carrier_fisher("/home/jupyter/CCDC27.tier1_0001.list","breast",replication_familial_breast)

In [ ]:
run_cancer_carrier_fisher("/home/jupyter/brat1_carriers.list", "pancancer",df_merged_v9)

In [ ]:
run_cancer_carrier_fisher(
    "/home/jupyter/BRCA1.tier1_0001.list",
    "breast",
    df_merged_v9[
        (df_merged_v9["sex_at_birth"] == "Female") &
        (
            (df_merged_v9["relatives_w_breast_cancer"].fillna(0) >= 1) |
            (df_merged_v9["pancancer_cancer"].fillna(0) == 0)
        )
    ]
)

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
from scipy.stats import zscore

def run_cancer_carrier_logistic(
    carrier_ids_file: str,
    cancer_type: str,
    df: pd.DataFrame
):
    cancer_col = f"{cancer_type}_cancer"

    # Load carrier IDs
    carrier_ids = set(
        pd.read_csv(carrier_ids_file, header=None, dtype=str)[0].str.strip()
    )

    # Prepare dataframe
    df = df.copy()
    if cancer_type in ["breast", "ovarian", "endometrial"]:
        df = df[df["sex_at_birth"] == "Female"]
    elif cancer_type in ["prostate","male_breast"]:
        df = df[df["sex_at_birth"] == "Male"]
    df["person_id"] = df["person_id"].astype(str).str.strip()

    df["case"] = (df[cancer_col] == 1).astype(int)
    df = df[(df[cancer_col] == 1) | (df["pancancer_cancer"] != 1)]
    df["carrier"] = df["person_id"].isin(carrier_ids).astype(int)
    df[[f"PC{i}" for i in range(1, 6)]] = df[[f"PC{i}" for i in range(1, 6)]].apply(zscore)
    #df["age"] = pd.to_numeric(df["age"], errors="coerce").astype(float)
    #df = df.dropna(subset=["age"])
    # Logistic regression adjusted for principal components
    # X = df[["carrier", "PC1", "PC2", "PC3", "PC4", "PC5"]].copy()
    # Binary encode sex_at_birth
    df["sex_binary"] = df["sex_at_birth"].map({"Female": 0, "Male": 1})

    
    # Logistic regression adjusted for principal components and sex

    X = df[["carrier","PC1","PC2","PC3","PC4","PC5"]].copy()
    X = sm.add_constant(X)
    y = df["case"]
    
    model = sm.Logit(y, X).fit(disp=False)

    print(f"Cancer type: {cancer_type}")
    print(f"Cases: {df['case'].sum():,}")
    print(f"Controls: {(df['case'] == 0).sum():,}")
    print(f"Carriers: {df['carrier'].sum():,}")
    print(f"Carrier OR: {np.exp(model.params['carrier']):.4f}")
    print(f"Carrier p-value: {model.pvalues['carrier']:.4e}")

    return

In [ ]:
run_cancer_carrier_logistic("/home/jupyter/CCDC27.tier1_0001.list","breast",replication_breast)

In [ ]:
tmp_df = df_merged_v9[
    ((df_merged_v9["num_relatives_w_cancer"].fillna(0) >= 3) & (df_merged_v9["pancancer_cancer"] == 1)) | (df_merged_v9["pancancer_cancer"].fillna(0) == 0) 
    ]

# carrier_ids = pd.read_csv(
#     "/home/jupyter/pathogenic_variants_carriers_in_AoU_CDRv9.list.tsv",
#     sep="\t",
#     header=None,
#     dtype=str
# )[0].dropna()

# tmp_df = df_merged_v9[
#     (
#         (
#             (df_merged_v9["num_relatives_w_cancer"].fillna(0) >= 3)
#             & (df_merged_v9["pancancer_cancer"] == 1)
#         )
#         | (df_merged_v9["pancancer_cancer"].fillna(0) == 0) # optional to ascertain by FHx for controls
#     )
#     & (~df_merged_v9["person_id"].astype(str).isin(carrier_ids))
# ]

run_cancer_carrier_logistic("/home/jupyter/CCDC27.tier1_0001.list","breast",tmp_df)

In [ ]:
run_cancer_carrier_logistic("/home/jupyter/brat1_carriers.list","pancancer",df_merged_v9)